# WSJ 2027 - Gruppindelning

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex should be as evenly spread as possible

## Algorithm
1. Assign lat/lng from geocode cache (kår → coordinates)
2. Sort by Hilbert curve (2D → 1D geographic ordering)
3. Cut into groups of 36
4. Swap optimization: fix friends → fix kår limits → improve diversity

## Output
- Group assignments with quality metrics
- CSV export: `wsj27_grupper.csv`
- KeplerGL map with group colors

In [ ]:
# Cell 1: Imports and API configuration
import requests
import json
import pandas as pd
import numpy as np
import os
from collections import defaultdict, Counter
from datetime import date

# Load credentials from .env file
from dotenv import load_dotenv
load_dotenv('/config/notebooks/wsj27/.env')

SCOUTNET_API_ID = os.environ['SCOUTNET_API_ID']
SCOUTNET_API_KEY = os.environ['SCOUTNET_API_KEY']
SCOUTNET_URL = "https://www.scoutnet.se/api/project/get/participants"

# Questions form API (for discovering question IDs)
QUESTIONS_API_KEY = os.environ['SCOUTNET_QUESTIONS_API_KEY']
QUESTIONS_URL = "https://www.scoutnet.se/api/project/get/questions"

WSJ_START = date(2027, 7, 29)
GROUP_SIZE = 36

# Fee categories
DELTAGARE_FEES = {'27561', '25694'}  # direktresa, rundresa
DELTAGARE_RUNDRESA = '25694'
DELTAGARE_DIREKTRESA = '27561'
IST_FEES = {'25702', '25696', '25697', '25693'}

# Question IDs (from Scoutnet form 39188)
Q_FRIEND_1_MEMBER_NO = '87660'  # "Medlemsnummer person 1"
Q_FRIEND_1_NAME = '87662'       # "Person 1" (free text name)
Q_FRIEND_2_MEMBER_NO = '87663'  # "Medlemsnummer person 2"
Q_FRIEND_2_NAME = '87665'       # "Person 2" (free text name)

print("API configuration loaded")

In [2]:
# Cell 2: Fetch and parse participants
response = requests.get(SCOUTNET_URL, auth=(SCOUTNET_API_ID, SCOUTNET_API_KEY))
response.raise_for_status()
raw_data = response.json()
participants_raw = raw_data.get('participants', {})

cancelled = sum(1 for p in participants_raw.values() if p.get('cancelled'))
confirmed = sum(1 for p in participants_raw.values() if p.get('confirmed') and not p.get('cancelled'))
unconfirmed = sum(1 for p in participants_raw.values() if not p.get('confirmed') and not p.get('cancelled'))

print(f"Fetched {len(participants_raw)} participants")
print(f"Confirmed: {confirmed}, Unconfirmed: {unconfirmed}, Cancelled: {cancelled}")

Fetched 1184 participants
Confirmed: 1127, Unconfirmed: 24, Cancelled: 33


In [3]:
# Cell 2.5: Inspect specific participant raw data
import json

member_no = '3324845'  # Walter Rappe White
p = participants_raw.get(member_no, {})
print(f'=== {p.get("first_name")} {p.get("last_name")} (member {member_no}) ===')
print(json.dumps(p, indent=2, ensure_ascii=False, default=str))

=== Walter Rappe White (member 3324845) ===
{
  "checked_in": false,
  "attended": false,
  "cancelled": false,
  "confirmed": true,
  "confirmed_at": "2026-02-24 17:47:00",
  "member_status": 2,
  "member_no": 3324845,
  "group_registration": false,
  "first_name": "Walter",
  "last_name": "Rappe White",
  "ssno": "8295",
  "registration_date": "2026-02-22 23:34:59",
  "cancelled_date": null,
  "sex": "1",
  "date_of_birth": "2012-03-26",
  "primary_email": "billywhite43@hotmail.com",
  "fee_id": 25694,
  "primary_membership_info": [],
  "group_registration_info": {
    "group_id": null,
    "patrol_id": null,
    "group_name": null,
    "org_id": null,
    "org_name": null,
    "district_id": null,
    "district_name": null,
    "patrol_name": null
  },
  "group_id": "Deprecated - use group_registration_info",
  "patrol_id": "Deprecated - use group_registration_info",
  "group_name": "Deprecated - use group_registration_info",
  "org_id": "Deprecated - use group_registration_info",
 

In [4]:
# Cell 3: Build participant DataFrame with age validation

def exact_age(birth_date, ref_date):
    """Calculate exact age in whole years at ref_date."""
    years = ref_date.year - birth_date.year
    if (ref_date.month, ref_date.day) < (birth_date.month, birth_date.day):
        years -= 1
    return years

def get_question(questions, qid):
    """Safely get a question answer, handling list/dict."""
    if not isinstance(questions, dict):
        return ''
    val = questions.get(qid, '')
    if val is None:
        return ''
    return str(val).strip()

rows = []
skipped = []
skipped_unconfirmed = 0

for mid, p in participants_raw.items():
    if p.get('cancelled'):
        continue
    
    # Skip unconfirmed registrations
    if not p.get('confirmed'):
        skipped_unconfirmed += 1
        continue
    
    fee_id = str(p.get('fee_id', ''))
    dob = p.get('date_of_birth', '')
    name = f"{p.get('first_name', '')} {p.get('last_name', '')}"
    
    birth = None
    if dob:
        try:
            birth = date.fromisoformat(dob)
        except ValueError:
            pass
    
    # Age validation
    if birth is None:
        skipped.append(f"  NO DOB: {name} fee={fee_id}")
        continue
    
    age = exact_age(birth, WSJ_START)
    
    if fee_id in DELTAGARE_FEES and (age < 14 or age >= 18):
        skipped.append(f"  DELTAGARE wrong age: {name} born {dob} (age {age})")
        continue
    if fee_id in IST_FEES and age < 18:
        skipped.append(f"  IST too young: {name} born {dob} (age {age})")
        continue
    
    # Determine category
    if fee_id in DELTAGARE_FEES:
        category = 'deltagare'
    elif fee_id in IST_FEES:
        category = 'ist'
    else:
        category = 'deltagare' if 14 <= age <= 17 else 'ist'
    
    # Extract kår info
    membership = p.get('primary_membership_info', {})
    if isinstance(membership, dict):
        kar = membership.get('group_name', '')
        district = membership.get('district_name', '')
        region = membership.get('region_name', '')
    else:
        kar, district, region = '', '', ''
    
    # Extract friend wishes from questions
    questions = p.get('questions', {})
    friend_1 = get_question(questions, Q_FRIEND_1_MEMBER_NO)
    friend_2 = get_question(questions, Q_FRIEND_2_MEMBER_NO)
    friend_1_name = get_question(questions, Q_FRIEND_1_NAME)
    friend_2_name = get_question(questions, Q_FRIEND_2_NAME)
    
    # Clean friend member numbers (ignore '0' and empty)
    friend_1 = friend_1 if friend_1 and friend_1 != '0' else ''
    friend_2 = friend_2 if friend_2 and friend_2 != '0' else ''
    
    rows.append({
        'member_no': str(p.get('member_no', mid)),
        'name': name,
        'birth_date': dob,
        'age': age,
        'sex': p.get('sex', 0),
        'fee_id': fee_id,
        'category': category,
        'travel': 'rundresa' if fee_id == DELTAGARE_RUNDRESA else ('direktresa' if fee_id == DELTAGARE_DIREKTRESA else 'other'),
        'kar': kar,
        'district': district,
        'region': region,
        'friend_1': friend_1,
        'friend_2': friend_2,
        'friend_1_name': friend_1_name,
        'friend_2_name': friend_2_name,
        'group': None,  # To be assigned
    })

df = pd.DataFrame(rows)

print(f"Total confirmed participants: {len(df)}")
print(f"Skipped: {skipped_unconfirmed} unconfirmed, {len(skipped)} wrong age/no DOB")
print(f"\nBy category:")
print(df['category'].value_counts().to_string())
print(f"\nBy travel type:")
print(df['travel'].value_counts().to_string())
print(f"\nFriend wishes:")
print(f"  With friend 1 member no: {(df['friend_1'] != '').sum()}")
print(f"  With friend 2 member no: {(df['friend_2'] != '').sum()}")
print(f"  With friend 1 name (text): {(df['friend_1_name'] != '').sum()}")
print(f"  With friend 2 name (text): {(df['friend_2_name'] != '').sum()}")

if skipped:
    print(f"\nSkipped participants:")
    for s in skipped:
        print(s)

Total confirmed participants: 1127
Skipped: 24 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    944
ist          183

By travel type:
travel
rundresa      762
other         183
direktresa    182

Friend wishes:
  With friend 1 member no: 617
  With friend 2 member no: 350
  With friend 1 name (text): 56
  With friend 2 name (text): 37


In [5]:
# Cell 4: Filter to rundresa + assign coordinates from geocode cache

GEOCODE_CACHE = '/config/notebooks/wsj27/scoutkar_geocode_cache.json'
with open(GEOCODE_CACHE, 'r', encoding='utf-8') as f:
    geocode_cache = json.load(f)

df_rundresa = df[df['travel'] == 'rundresa'].copy().reset_index(drop=True)

# Assign lat/lng from geocode cache via kår name
def get_coords(kar_name):
    geo = geocode_cache.get(kar_name, {})
    return geo.get('lat'), geo.get('lng')

df_rundresa['lat'] = df_rundresa['kar'].apply(lambda k: get_coords(k)[0])
df_rundresa['lng'] = df_rundresa['kar'].apply(lambda k: get_coords(k)[1])

# Sweden centroid for participants without coordinates
SWEDEN_LAT, SWEDEN_LNG = 62.0, 15.0
no_coords = df_rundresa['lat'].isna().sum()
df_rundresa['lat'] = df_rundresa['lat'].fillna(SWEDEN_LAT)
df_rundresa['lng'] = df_rundresa['lng'].fillna(SWEDEN_LNG)

n_full_groups = len(df_rundresa) // GROUP_SIZE
remainder = len(df_rundresa) % GROUP_SIZE
total_groups = n_full_groups + (1 if remainder > 0 else 0)

print(f"=== Phase 1: Deltagare rundresa ===")
print(f"Participants: {len(df_rundresa)}")
print(f"With coordinates: {len(df_rundresa) - no_coords}")
print(f"Without coordinates (Sweden centroid): {no_coords}")
print(f"Groups: {n_full_groups} x 36 + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_rundresa['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_rundresa['age'].value_counts().sort_index().to_string())
sex_map = {'0': 'Okänt', '1': 'Man', '2': 'Kvinna', '3': 'Annat', '4': 'Icke-binär',
           0: 'Okänt', 1: 'Man', 2: 'Kvinna', 3: 'Annat', 4: 'Icke-binär'}
print(f"\nBy sex:")
print(df_rundresa['sex'].map(sex_map).value_counts().to_string())

=== Phase 1: Deltagare rundresa ===
Participants: 762
With coordinates: 741
Without coordinates (Sweden centroid): 21
Groups: 21 x 36 + 1 x 6 = 22 total

By region:
region
Region Stockholm    228
Region Södra        190
Region Västra       151
Region Norr/Mitt     75
Region Östra         52
                     21

By age:
age
14    189
15    225
16    178
17    170

By sex:
sex
Kvinna    386
Man       372
Annat       4


In [6]:
# Cell 5: Friend wish analysis (runs before text matching)
# Build friend graph: member_no -> [friend_1, friend_2]

member_set = set(df_rundresa['member_no'].values)
all_members = set(df['member_no'].values)

# Count text-only wishes before resolving
text_only_count = sum(1 for _, r in df_rundresa.iterrows()
                      if (not r['friend_1'] and r['friend_1_name']) or
                         (not r['friend_2'] and r['friend_2_name']))

# Summary before text matching
print(f"=== Friend Wishes (before text matching) ===")
print(f"With friend 1 (member no): {(df_rundresa['friend_1'] != '').sum()}")
print(f"With friend 2 (member no): {(df_rundresa['friend_2'] != '').sum()}")
print(f"With text name only (no member no): {text_only_count}")
print(f"Without any friend wish: {((df_rundresa['friend_1'] == '') & (df_rundresa['friend_2'] == '') & (df_rundresa['friend_1_name'] == '') & (df_rundresa['friend_2_name'] == '')).sum()}")

=== Friend Wishes (before text matching) ===
With friend 1 (member no): 497
With friend 2 (member no): 294
With text name only (no member no): 37
Without any friend wish: 229


In [7]:
# Cell 5b: Resolve text-only friend wishes via name matching
import re
from difflib import SequenceMatcher

def normalize_name(name):
    return re.sub(r'\s+', ' ', name.strip().lower())

def parse_friend_text(text):
    """Parse free-text friend wish into (name_part, kar_hint)."""
    text = text.strip()
    # Skip generic wishes (not a specific person)
    skip_patterns = ['gärna någon', 'övriga deltagare', 'scouter från']
    if any(p in text.lower() for p in skip_patterns):
        return None, text
    
    kar_hint = ''
    name_part = text
    
    if ',' in text:
        parts = [p.strip() for p in text.split(',')]
        name_part = parts[0]
        kar_hint = ' '.join(parts[1:])
    elif ' - ' in text:
        parts = [p.strip() for p in text.split(' - ', 1)]
        name_part = parts[0]
        kar_hint = parts[1]
    else:
        kar_starters = ['scoutkår', 'sjöscoutkår', 'scouterkår', 'scoutskår',
                        'equmenia', 'kåren', 'scoutkåren']
        text_lower = text.lower()
        for kw in kar_starters:
            idx = text_lower.find(kw)
            if idx > 0:
                before = text[:idx].rstrip()
                bwords = before.split()
                if len(bwords) > 2:
                    name_part = ' '.join(bwords[:2])
                    kar_hint = ' '.join(bwords[2:]) + ' ' + text[idx:]
                    kar_hint = kar_hint.strip()
                break
        else:
            words = text.split()
            if len(words) > 3:
                name_part = ' '.join(words[:2])
                kar_hint = ' '.join(words[2:])
    
    return name_part.strip(), kar_hint.strip()

def fuzzy_match_name(query_name, name_lookup, kar_hint='', threshold=0.75):
    """Find best matching participant by name with optional kår boost."""
    query_norm = normalize_name(query_name)
    
    # 1. Exact match
    if query_norm in name_lookup:
        matches = name_lookup[query_norm]
        if len(matches) == 1:
            return matches[0], 'exact', 1.0
        if kar_hint:
            kar_lower = kar_hint.lower()
            for m in matches:
                if kar_lower in m['kar'].lower() or m['kar'].lower() in kar_lower:
                    return m, 'exact+kår', 1.0
        return matches[0], 'exact(ambiguous)', 1.0
    
    # 2. Try first + last word
    query_words = query_norm.split()
    if len(query_words) >= 2:
        short_query = query_words[0] + ' ' + query_words[-1]
        if short_query in name_lookup:
            matches = name_lookup[short_query]
            if len(matches) == 1:
                return matches[0], 'first+last', 0.95
    
    # 3. Fuzzy matching
    best_score = 0
    best_match = None
    for norm_name, candidates in name_lookup.items():
        score = SequenceMatcher(None, query_norm, norm_name).ratio()
        cand_words = norm_name.split()
        if len(cand_words) >= 2:
            short_cand = cand_words[0] + ' ' + cand_words[-1]
            score = max(score, SequenceMatcher(None, query_norm, short_cand).ratio())
        if len(query_words) >= 2:
            short_q = query_words[0] + ' ' + query_words[-1]
            score = max(score, SequenceMatcher(None, short_q, norm_name).ratio())
        if score > best_score:
            best_score = score
            best_match = candidates[0]
    
    if best_score >= threshold and best_match:
        method = f'fuzzy({best_score:.2f})'
        if kar_hint:
            kar_lower = kar_hint.lower()
            if kar_lower in best_match['kar'].lower() or best_match['kar'].lower() in kar_lower:
                method += '+kår'
                best_score = min(best_score + 0.1, 1.0)
        return best_match, method, best_score
    
    return None, 'no_match', best_score

# Build name lookup
name_lookup = defaultdict(list)
for _, row in df.iterrows():
    norm = normalize_name(row['name'])
    name_lookup[norm].append({
        'member_no': row['member_no'], 'name': row['name'],
        'kar': row['kar'], 'travel': row['travel'],
    })

# Collect text-only wishes
text_wishes = []
for _, row in df_rundresa.iterrows():
    if not row['friend_1'] and row['friend_1_name']:
        text_wishes.append({
            'wisher_member_no': row['member_no'], 'wisher': row['name'],
            'friend_text': row['friend_1_name'], 'slot': 'friend_1',
        })
    if not row['friend_2'] and row['friend_2_name']:
        text_wishes.append({
            'wisher_member_no': row['member_no'], 'wisher': row['name'],
            'friend_text': row['friend_2_name'], 'slot': 'friend_2',
        })

# Match and validate
verified = []
unresolved = []
skipped_generic = []

for tw in text_wishes:
    name_part, kar_hint = parse_friend_text(tw['friend_text'])
    if name_part is None:
        skipped_generic.append(tw)
        continue
    
    match, method, score = fuzzy_match_name(name_part, name_lookup, kar_hint)
    
    if match:
        # Validate: first or last name must match
        parsed_words = normalize_name(name_part).split()
        match_words = normalize_name(match['name']).split()
        if score < 0.90:
            p_first, p_last = parsed_words[0], parsed_words[-1] if len(parsed_words) > 1 else ''
            m_first, m_last = match_words[0], match_words[-1] if len(match_words) > 1 else ''
            if p_first != m_first and p_last != m_last:
                unresolved.append({**tw, 'reason': f'name mismatch ({match["name"]})'})
                continue
        verified.append({**tw, 'match': match, 'method': method, 'score': score})
    else:
        unresolved.append({**tw, 'reason': 'no match found'})

# Apply matches to DataFrame
updates = 0
for v in verified:
    idx = df_rundresa[df_rundresa['member_no'] == v['wisher_member_no']].index
    if len(idx) > 0 and df_rundresa.at[idx[0], v['slot']] == '':
        df_rundresa.at[idx[0], v['slot']] = v['match']['member_no']
        # Also update main df
        main_idx = df[df['member_no'] == v['wisher_member_no']].index
        if len(main_idx) > 0:
            df.at[main_idx[0], v['slot']] = v['match']['member_no']
        updates += 1

print(f"=== Text Friend Name Matching ===")
print(f"Text-only wishes: {len(text_wishes)}")
print(f"Matched & applied: {len(verified)}")
print(f"Generic wishes (not a person): {len(skipped_generic)}")
print(f"Unresolved (friend not in project): {len(unresolved)}")

print(f"\nMatched:")
for v in sorted(verified, key=lambda x: x['wisher']):
    m = v['match']
    print(f"  {v['wisher']} -> {m['name']} ({m['kar']}) [{m['travel']}] via {v['method']}")

if unresolved:
    print(f"\nUnresolved:")
    for u in sorted(unresolved, key=lambda x: x['wisher']):
        print(f"  {u['wisher']} -> \"{u['friend_text']}\" ({u['reason']})")

if skipped_generic:
    print(f"\nGeneric wishes (skipped):")
    for s in skipped_generic:
        print(f"  {s['wisher']} -> \"{s['friend_text']}\"")

=== Text Friend Name Matching ===
Text-only wishes: 59
Matched & applied: 24
Generic wishes (not a person): 3
Unresolved (friend not in project): 32

Matched:
  Albin Åkesson -> Malte Lindsjö (Jonstorps Kustscoutkår) [rundresa] via fuzzy(0.78)
  Alex Liljerot Priftis -> Vera Tollwé (Årsta Scoutkår) [rundresa] via fuzzy(0.86)
  Alma Stössel Weinryb -> Lotten Hellman (Södermalms Scoutkår) [rundresa] via exact
  Alma Stössel Weinryb -> Alfons Ekelin Sintorn (Södermalms Scoutkår) [rundresa] via fuzzy(0.76)+kår
  Charlie Lindberg -> Axel Lindroth (Saltsjö-Boo Scoutkår) [rundresa] via fuzzy(0.96)
  Elsie Stenfeldt -> Hugo Johannessen (Tuve Scoutkår) [rundresa] via exact
  Emilio Larsen -> Emmy Nilsson (Glimåkra Scoutkår) [rundresa] via fuzzy(0.77)
  Flora Wiklander -> Oscar Löf (Scoutkåren Gustaf Vasa-Bredäng) [direktresa] via exact
  Ida Jönsson -> Vendela Gustafsson (Huddinge Scoutkår) [rundresa] via fuzzy(0.76)
  Linnea Barath -> Erik Hartwig (Staffanstorps Scoutkår Torparna) [rundresa] v

In [8]:
# Cell 6: Final friend wish summary (after text matching)

member_set = set(df_rundresa['member_no'].values)
all_members = set(df['member_no'].values)

friend_wishes = {}
for _, row in df_rundresa.iterrows():
    wishes = []
    if row['friend_1'] and row['friend_1'] in member_set:
        wishes.append(row['friend_1'])
    if row['friend_2'] and row['friend_2'] in member_set:
        wishes.append(row['friend_2'])
    if wishes:
        friend_wishes[row['member_no']] = wishes

# Mutual pairs
mutual_pairs = set()
one_way = []
for member, wishes in friend_wishes.items():
    for friend in wishes:
        if friend in friend_wishes and member in friend_wishes[friend]:
            pair = tuple(sorted([member, friend]))
            mutual_pairs.add(pair)
        else:
            one_way.append((member, friend))

# Cross-category and unknown
cross_category = []
unknown_member = []
for _, row in df_rundresa.iterrows():
    for f_col in ['friend_1', 'friend_2']:
        fid = row[f_col]
        if fid and fid not in member_set:
            if fid in all_members:
                friend_row = df[df['member_no'] == fid].iloc[0]
                cross_category.append((row['name'], friend_row['name'], friend_row['travel']))
            else:
                unknown_member.append((row['name'], fid))

print(f"=== Final Friend Wish Summary (after text matching) ===")
print(f"Participants with >=1 wish in rundresa: {len(friend_wishes)}")
print(f"Mutual pairs (both wish each other): {len(mutual_pairs)}")
print(f"One-way wishes: {len(one_way)}")
print(f"Without any resolved friend wish: {len(df_rundresa) - len(friend_wishes)}")
print(f"Cross-category wishes: {len(cross_category)}")
print(f"Unknown member numbers: {len(unknown_member)}")

if cross_category:
    print(f"\nCross-category ({len(cross_category)}):")
    for name, fname, ftravel in cross_category[:10]:
        print(f"  {name} -> {fname} ({ftravel})")
    if len(cross_category) > 10:
        print(f"  ... and {len(cross_category)-10} more")

=== Final Friend Wish Summary (after text matching) ===
Participants with >=1 wish in rundresa: 428
Mutual pairs (both wish each other): 218
One-way wishes: 171
Without any resolved friend wish: 334
Cross-category wishes: 31
Unknown member numbers: 183

Cross-category (31):
  Signe Andersson -> Tove Hermansson (other)
  William Arlehamnn -> Helmer Palmblad (direktresa)
  Kajsa Berg -> Maxine Danielsson (direktresa)
  Alma Bjerre -> Meya Björkman (direktresa)
  Alma Bjerre -> Ella Lundberg (direktresa)
  Emilia Elm -> Olivia Wernås (direktresa)
  Lukas Eriksson -> Mika Bruzelius (direktresa)
  Lukas Eriksson -> Minja Bruzelius (direktresa)
  Olivia Fisler Nord -> Tilde Nargell (direktresa)
  Olivia Fisler Nord -> Olivia Limin (direktresa)
  ... and 21 more


In [9]:
# Cell 7: Group Assignment Algorithm
import random
import math
random.seed(42)

# ===========================================================================
# Hilbert curve: maps 2D lat/lng to 1D index preserving spatial locality
# ===========================================================================
def hilbert_xy2d(n, x, y):
    """Convert (x,y) to Hilbert curve distance in n x n grid."""
    d = 0
    s = n // 2
    while s > 0:
        rx = 1 if (x & s) > 0 else 0
        ry = 1 if (y & s) > 0 else 0
        d += s * s * ((3 * rx) ^ ry)
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        s //= 2
    return d

HILBERT_N = 256
LAT_RANGE = (55.0, 70.0)
LNG_RANGE = (10.0, 25.0)

def geo_to_hilbert(lat, lng):
    x = int((lat - LAT_RANGE[0]) / (LAT_RANGE[1] - LAT_RANGE[0]) * (HILBERT_N - 1))
    y = int((lng - LNG_RANGE[0]) / (LNG_RANGE[1] - LNG_RANGE[0]) * (HILBERT_N - 1))
    x = max(0, min(HILBERT_N - 1, x))
    y = max(0, min(HILBERT_N - 1, y))
    return hilbert_xy2d(HILBERT_N, x, y)

# Compute Hilbert index and sort
df_rundresa['hilbert'] = df_rundresa.apply(
    lambda r: geo_to_hilbert(r['lat'], r['lng']), axis=1)
df_sorted = df_rundresa.sort_values('hilbert').reset_index(drop=True)

# ===========================================================================
# Fast lookup arrays (avoid pandas overhead in hot loops)
# ===========================================================================
n = len(df_sorted)
lats = df_sorted['lat'].values.copy()
lngs = df_sorted['lng'].values.copy()
kars_arr = df_sorted['kar'].values.copy()
ages_arr = df_sorted['age'].values.copy()
sexes_arr = df_sorted['sex'].values.copy()
member_arr = df_sorted['member_no'].values.copy()
f1_arr = df_sorted['friend_1'].values.copy()
f2_arr = df_sorted['friend_2'].values.copy()
member_to_idx = {m: i for i, m in enumerate(member_arr)}
rundresa_set = set(member_arr)

# Reverse friend index: who depends on each participant for their friend wish
friend_of = defaultdict(set)
for i in range(n):
    if f1_arr[i] and f1_arr[i] in rundresa_set:
        friend_of[f1_arr[i]].add(i)
    if f2_arr[i] and f2_arr[i] in rundresa_set:
        friend_of[f2_arr[i]].add(i)

# Group assignment array
group_of = np.zeros(n, dtype=int)
for i in range(n_full_groups):
    group_of[i * GROUP_SIZE:(i + 1) * GROUP_SIZE] = i
if remainder > 0:
    group_of[n_full_groups * GROUP_SIZE:] = n_full_groups

MAX_KAR = 8

# ===========================================================================
# Helper functions
# ===========================================================================
def get_group_members(g):
    return np.where(group_of == g)[0]

def has_friend_wish(idx):
    f1, f2 = f1_arr[idx], f2_arr[idx]
    return bool((f1 and f1 in rundresa_set) or (f2 and f2 in rundresa_set))

def friend_satisfied(idx):
    g = group_of[idx]
    gm = set(member_arr[get_group_members(g)])
    f1, f2 = f1_arr[idx], f2_arr[idx]
    return (f1 in gm) or (f2 in gm)

def kar_count_in_group(g, kar):
    gm = get_group_members(g)
    return sum(1 for i in gm if kars_arr[i] == kar)

def can_swap(i1, i2):
    """Check if swap respects kår limit (max 8)."""
    g1, g2 = group_of[i1], group_of[i2]
    if g1 == g2:
        return False
    k1, k2 = kars_arr[i1], kars_arr[i2]
    if k1 == k2:
        return True
    if k2 and kar_count_in_group(g1, k2) - (1 if k2 == k1 else 0) + 1 > MAX_KAR:
        return False
    if k1 and kar_count_in_group(g2, k1) - (1 if k1 == k2 else 0) + 1 > MAX_KAR:
        return False
    return True

def do_swap(i1, i2):
    group_of[i1], group_of[i2] = group_of[i2], group_of[i1]

def count_friend_satisfied():
    return sum(1 for i in range(n) if has_friend_wish(i) and friend_satisfied(i))

def count_friend_total():
    return sum(1 for i in range(n) if has_friend_wish(i))

def count_kar_violations():
    total = 0
    for g in range(total_groups):
        gm = get_group_members(g)
        counts = Counter(kars_arr[i] for i in gm if kars_arr[i])
        total += sum(max(0, c - MAX_KAR) for c in counts.values())
    return total

def affected_by_swap(i1, i2):
    """Get all participants whose friend satisfaction could change from a swap."""
    affected = {i1, i2}
    m1, m2 = member_arr[i1], member_arr[i2]
    affected.update(friend_of.get(m1, set()))
    affected.update(friend_of.get(m2, set()))
    return affected

def geo_dist_sq(i1, i2):
    """Squared geographic distance (fast, no sqrt needed for comparison)."""
    return (lats[i1] - lats[i2])**2 + (lngs[i1] - lngs[i2])**2

def group_geo_spread(g):
    """Mean squared distance to group centroid (geographic compactness)."""
    gm = get_group_members(g)
    if len(gm) <= 1:
        return 0.0
    clat = np.mean(lats[gm])
    clng = np.mean(lngs[gm])
    return np.mean([(lats[i] - clat)**2 + (lngs[i] - clng)**2 for i in gm])

# ===========================================================================
# Phase 1: Initial geographic assignment (already done by sort + cut)
# ===========================================================================
friend_total = count_friend_total()
print("=== Phase 1: Geographic sort + cut ===")
print(f"  Friend satisfaction: {count_friend_satisfied()}/{friend_total}")
print(f"  Kår violations: {count_kar_violations()}")
print(f"  Avg geo spread: {np.mean([group_geo_spread(g) for g in range(total_groups)]):.4f}")

# ===========================================================================
# Phase 2: Fix friend wishes via targeted swaps
# ===========================================================================
print("\n=== Phase 2: Fix friend wishes ===")
friend_swaps = 0
for idx in range(n):
    if not has_friend_wish(idx) or friend_satisfied(idx):
        continue
    
    target_groups = set()
    for fid in [f1_arr[idx], f2_arr[idx]]:
        if fid and fid in member_to_idx:
            target_groups.add(group_of[member_to_idx[fid]])
    target_groups.discard(group_of[idx])
    if not target_groups:
        continue
    
    best_cidx = None
    best_net = -999
    best_dist = float('inf')
    for tg in target_groups:
        for cidx in get_group_members(tg):
            if not can_swap(idx, cidx):
                continue
            affected = affected_by_swap(idx, cidx)
            old_sat = sum(1 for a in affected if has_friend_wish(a) and friend_satisfied(a))
            do_swap(idx, cidx)
            new_sat = sum(1 for a in affected if has_friend_wish(a) and friend_satisfied(a))
            do_swap(idx, cidx)  # undo
            net = new_sat - old_sat
            dist = geo_dist_sq(idx, cidx)
            # Prefer higher net friend gain, then closer geographically
            if net > best_net or (net == best_net and dist < best_dist):
                best_net = net
                best_cidx = cidx
                best_dist = dist
    
    if best_cidx is not None and best_net >= 0:
        do_swap(idx, best_cidx)
        friend_swaps += 1

print(f"  Swaps: {friend_swaps}")
print(f"  Friend satisfaction: {count_friend_satisfied()}/{friend_total}")
print(f"  Kår violations: {count_kar_violations()}")
print(f"  Avg geo spread: {np.mean([group_geo_spread(g) for g in range(total_groups)]):.4f}")

# ===========================================================================
# Phase 3: Fix kår violations (prefer geographically closest swap)
# ===========================================================================
print("\n=== Phase 3: Fix kår violations (geo-aware) ===")
kar_swaps = 0
for g in range(total_groups):
    gm = get_group_members(g)
    counts = Counter(kars_arr[i] for i in gm if kars_arr[i])
    for kar, cnt in counts.items():
        if cnt <= MAX_KAR:
            continue
        excess = [i for i in gm if kars_arr[i] == kar]
        for idx in excess[MAX_KAR:]:
            # Collect ALL valid swap candidates across all other groups
            candidates = []
            for og in range(total_groups):
                if og == g:
                    continue
                for cidx in get_group_members(og):
                    if kars_arr[cidx] == kar:
                        continue
                    if can_swap(idx, cidx):
                        candidates.append(cidx)
            if not candidates:
                continue
            # Pick the geographically closest candidate
            best_cidx = min(candidates, key=lambda c: geo_dist_sq(idx, c))
            do_swap(idx, best_cidx)
            kar_swaps += 1

print(f"  Swaps: {kar_swaps}")
print(f"  Kår violations: {count_kar_violations()}")
print(f"  Friend satisfaction: {count_friend_satisfied()}/{friend_total}")
print(f"  Avg geo spread: {np.mean([group_geo_spread(g) for g in range(total_groups)]):.4f}")

# ===========================================================================
# Phase 2b: Re-fix friend wishes lost in Phase 3
# ===========================================================================
print("\n=== Phase 2b: Re-fix friends after kår fix ===")
friend_swaps_2b = 0
for idx in range(n):
    if not has_friend_wish(idx) or friend_satisfied(idx):
        continue
    
    target_groups = set()
    for fid in [f1_arr[idx], f2_arr[idx]]:
        if fid and fid in member_to_idx:
            target_groups.add(group_of[member_to_idx[fid]])
    target_groups.discard(group_of[idx])
    if not target_groups:
        continue
    
    best_cidx = None
    best_net = -999
    best_dist = float('inf')
    for tg in target_groups:
        for cidx in get_group_members(tg):
            if not can_swap(idx, cidx):
                continue
            affected = affected_by_swap(idx, cidx)
            old_sat = sum(1 for a in affected if has_friend_wish(a) and friend_satisfied(a))
            do_swap(idx, cidx)
            new_sat = sum(1 for a in affected if has_friend_wish(a) and friend_satisfied(a))
            do_swap(idx, cidx)  # undo
            net = new_sat - old_sat
            dist = geo_dist_sq(idx, cidx)
            if net > best_net or (net == best_net and dist < best_dist):
                best_net = net
                best_cidx = cidx
                best_dist = dist
    
    if best_cidx is not None and best_net >= 0:
        do_swap(idx, best_cidx)
        friend_swaps_2b += 1

print(f"  Swaps: {friend_swaps_2b}")
print(f"  Friend satisfaction: {count_friend_satisfied()}/{friend_total}")
print(f"  Kår violations: {count_kar_violations()}")
print(f"  Avg geo spread: {np.mean([group_geo_spread(g) for g in range(total_groups)]):.4f}")

# ===========================================================================
# Phase 4: Diversity optimization (simulated annealing)
# Preserves friend satisfaction AND geographic compactness
# ===========================================================================
print("\n=== Phase 4: Diversity optimization (geo-penalized) ===")

def group_diversity(g):
    """Diversity score: age entropy + sex entropy (higher = more diverse)."""
    gm = get_group_members(g)
    if len(gm) == 0:
        return 0
    age_c = Counter(ages_arr[i] for i in gm)
    total = sum(age_c.values())
    age_ent = -sum((c/total) * math.log2(c/total) for c in age_c.values() if c > 0)
    sex_c = Counter(sexes_arr[i] for i in gm)
    total_s = sum(sex_c.values())
    sex_ent = -sum((c/total_s) * math.log2(c/total_s) for c in sex_c.values() if c > 0)
    return age_ent + sex_ent

GEO_WEIGHT = 2.0  # How much to penalize geographic spread increase

div_before = sum(group_diversity(g) for g in range(total_groups))
geo_before = np.mean([group_geo_spread(g) for g in range(total_groups)])
diversity_swaps = 0
temperature = 1.0

for iteration in range(10000):
    i1 = random.randint(0, n - 1)
    i2 = random.randint(0, n - 1)
    g1, g2 = group_of[i1], group_of[i2]
    if g1 == g2 or not can_swap(i1, i2):
        continue
    
    # Check all affected participants' friend satisfaction
    affected = affected_by_swap(i1, i2)
    old_sat = sum(1 for a in affected if has_friend_wish(a) and friend_satisfied(a))
    
    old_div = group_diversity(g1) + group_diversity(g2)
    old_geo = group_geo_spread(g1) + group_geo_spread(g2)
    do_swap(i1, i2)
    
    new_sat = sum(1 for a in affected if has_friend_wish(a) and friend_satisfied(a))
    new_div = group_diversity(g1) + group_diversity(g2)
    new_geo = group_geo_spread(g1) + group_geo_spread(g2)
    
    # Combined score: diversity gain minus geographic spread penalty
    div_improvement = new_div - old_div
    geo_penalty = (new_geo - old_geo) * GEO_WEIGHT
    improvement = div_improvement - geo_penalty
    
    # Reject if any friend satisfaction is lost, or if combined score worsens (with SA)
    if new_sat < old_sat or (improvement < 0 and random.random() > math.exp(improvement / max(temperature, 0.01))):
        do_swap(i1, i2)  # reject
    else:
        diversity_swaps += 1
    
    temperature *= 0.9995

div_after = sum(group_diversity(g) for g in range(total_groups))
geo_after = np.mean([group_geo_spread(g) for g in range(total_groups)])
print(f"  Swaps: {diversity_swaps}")
print(f"  Diversity score: {div_before:.2f} -> {div_after:.2f}")
print(f"  Avg geo spread: {geo_before:.4f} -> {geo_after:.4f}")

# ===========================================================================
# Write results back to DataFrame
# ===========================================================================
df_sorted['group'] = group_of

print(f"\n{'='*50}")
print(f"=== FINAL RESULTS ===")
print(f"{'='*50}")
print(f"Groups: {n_full_groups} x 36 + 1 x {remainder}")
print(f"Friend satisfaction: {count_friend_satisfied()}/{friend_total} ({count_friend_satisfied()/max(1,friend_total)*100:.0f}%)")
print(f"Kår violations: {count_kar_violations()}")
print(f"Total swaps: {friend_swaps + kar_swaps + friend_swaps_2b + diversity_swaps}")
print(f"Diversity: {div_after:.2f}")
print(f"Avg geo spread: {geo_after:.4f}")

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 367/428
  Kår violations: 56
  Avg geo spread: 0.6452

=== Phase 2: Fix friend wishes ===
  Swaps: 42
  Friend satisfaction: 426/428
  Kår violations: 50
  Avg geo spread: 1.2847

=== Phase 3: Fix kår violations (geo-aware) ===
  Swaps: 47
  Kår violations: 0
  Friend satisfaction: 377/428
  Avg geo spread: 1.2901

=== Phase 2b: Re-fix friends after kår fix ===
  Swaps: 27
  Friend satisfaction: 425/428
  Kår violations: 0
  Avg geo spread: 1.3460

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 414
  Diversity score: 63.84 -> 65.04
  Avg geo spread: 1.3460 -> 1.6945

=== FINAL RESULTS ===
Groups: 21 x 36 + 1 x 6
Friend satisfaction: 425/428 (99%)
Kår violations: 0
Total swaps: 530
Diversity: 65.04
Avg geo spread: 1.6945


In [10]:
# Cell 8: Per-group quality metrics

sex_map = {0: 'Okänt', 1: 'Man', 2: 'Kvinna', 3: 'Annat', 4: 'Icke-binär',
           '0': 'Okänt', '1': 'Man', '2': 'Kvinna', '3': 'Annat', '4': 'Icke-binär'}

def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371
    dlat, dlng = math.radians(lat2 - lat1), math.radians(lng2 - lng1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlng/2)**2
    return R * 2 * math.asin(min(1.0, math.sqrt(a)))

print(f"{'Grupp':>5} {'Storlek':>7} {'Vän%':>5} {'MaxKår':>6} {'M/K/A':>9} "
      f"{'14':>3} {'15':>3} {'16':>3} {'17':>3} {'AvståndKm':>10} {'Kårer':>5}")
print("-" * 75)

for g in range(total_groups):
    gm = get_group_members(g)
    rows = df_sorted.loc[gm]
    
    # Size
    size = len(gm)
    
    # Friend satisfaction
    f_ok = sum(1 for i in gm if has_friend_wish(i) and friend_satisfied(i))
    f_tot = sum(1 for i in gm if has_friend_wish(i))
    f_pct = f"{f_ok}/{f_tot}" if f_tot > 0 else "-"
    
    # Max kår count
    kar_c = Counter(kars_arr[i] for i in gm if kars_arr[i])
    max_kar = max(kar_c.values()) if kar_c else 0
    
    # Sex distribution
    sex_c = Counter(sex_map.get(sexes_arr[i], '?') for i in gm)
    m_k_a = f"{sex_c.get('Man',0)}/{sex_c.get('Kvinna',0)}/{sex_c.get('Annat',0)}"
    
    # Age distribution
    age_c = Counter(ages_arr[i] for i in gm)
    
    # Geographic spread: mean distance to centroid
    clat, clng = np.mean(lats[gm]), np.mean(lngs[gm])
    avg_dist = np.mean([haversine_km(lats[i], lngs[i], clat, clng) for i in gm])
    
    # Unique kårer
    n_karer = len(kar_c)
    
    print(f"{g+1:>5} {size:>7} {f_pct:>5} {max_kar:>6} {m_k_a:>9} "
          f"{age_c.get(14,0):>3} {age_c.get(15,0):>3} {age_c.get(16,0):>3} {age_c.get(17,0):>3} "
          f"{avg_dist:>9.0f} {n_karer:>5}")

# Overall stats
print("-" * 75)
all_dists = []
for g in range(total_groups):
    gm = get_group_members(g)
    clat, clng = np.mean(lats[gm]), np.mean(lngs[gm])
    all_dists.append(np.mean([haversine_km(lats[i], lngs[i], clat, clng) for i in gm]))
print(f"Avg geographic spread: {np.mean(all_dists):.0f} km")
print(f"Min/Max spread: {np.min(all_dists):.0f} / {np.max(all_dists):.0f} km")

Grupp Storlek  Vän% MaxKår     M/K/A  14  15  16  17  AvståndKm Kårer
---------------------------------------------------------------------------
    1      36 23/24      7   14/22/0  10  13   7   6        69    16
    2      36 27/27      7   23/13/0  12   8   7   9        13    14
    3      36 22/22      6   13/23/0  14   9   7   6        34    18
    4      36 19/19      6   18/18/0  12  11   6   7        59    18
    5      36 19/19      6   20/16/0   9  10   7  10       105    20
    6      36 23/23      7   19/17/0   8  10   8  10        65    18
    7      36 23/23      8   12/24/0   9   9   8  10        45    11
    8      36 15/15      7   20/16/0   5   9  12  10       100    20
    9      36 24/24      6   11/25/0   6  12   8  10        52    14
   10      36 20/20      5   14/22/0   8   7  11  10       182    17
   11      36 20/20      7   17/18/1   6  11   9  10       121    15
   12      36 22/22      6   18/18/0   6  12  10   8        76    22
   13      36 15/15      5

In [11]:
# Cell 9: Export results to CSV

OUTPUT_DIR = '/config/notebooks/wsj27'

# Build export DataFrame
export_cols = ['group', 'name', 'member_no', 'age', 'sex', 'kar', 'district', 'region',
               'friend_1', 'friend_2', 'lat', 'lng']
df_export = df_sorted[export_cols].copy()
df_export['group'] = df_export['group'] + 1  # 1-indexed for humans
df_export['sex'] = df_export['sex'].map(sex_map)
df_export = df_export.sort_values(['group', 'kar', 'name']).reset_index(drop=True)

# Save CSV
csv_path = f'{OUTPUT_DIR}/wsj27_grupper.csv'
df_export.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"Saved {len(df_export)} participants to {csv_path}")

# Save JSON summary
group_summary = []
for g in range(total_groups):
    gm = get_group_members(g)
    kar_c = Counter(kars_arr[i] for i in gm if kars_arr[i])
    age_c = Counter(int(ages_arr[i]) for i in gm)
    sex_c = Counter(sex_map.get(sexes_arr[i], '?') for i in gm)
    clat, clng = float(np.mean(lats[gm])), float(np.mean(lngs[gm]))
    
    group_summary.append({
        'group': g + 1,
        'size': int(len(gm)),
        'centroid': {'lat': round(clat, 4), 'lng': round(clng, 4)},
        'age_distribution': dict(sorted(age_c.items())),
        'sex_distribution': dict(sex_c),
        'karer': dict(kar_c.most_common()),
        'members': [
            {'name': df_sorted.at[i, 'name'], 'member_no': str(member_arr[i]),
             'kar': str(kars_arr[i]), 'age': int(ages_arr[i])}
            for i in sorted(gm, key=lambda x: kars_arr[x])
        ]
    })

json_path = f'{OUTPUT_DIR}/wsj27_grupper.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(group_summary, f, ensure_ascii=False, indent=2)
print(f"Saved group summary to {json_path}")

# Preview
print(f"\nCSV preview (first 10 rows):")
print(df_export.head(10).to_string(index=False))

Saved 762 participants to /config/notebooks/wsj27/wsj27_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_grupper.json

CSV preview (first 10 rows):
 group              name member_no  age    sex                kar                   district       region friend_1 friend_2       lat       lng
     1    Joseph Alasady   3372969   15    Man                                                             3396330  3416313 62.000000 15.000000
     1         Anouk Asp   3351398   15 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra  3355911  3386859 55.664664 13.346159
     1        Freja Malm   3328985   15 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra  3336759  3329706 55.664664 13.346159
     1   Saga Lee Zander   3386859   14 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra  3351398  3355911 55.664664 13.346159
     1    Stella Jönsson   3355911   14 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra  3351398  

In [12]:
# Cell 10: KeplerGL map - participants colored by group
from keplergl import KeplerGl

# Build map DataFrame with small jitter so same-kår participants don't overlap
np.random.seed(42)
df_map = df_sorted[['name', 'kar', 'age', 'lat', 'lng', 'group']].copy()
df_map['group'] = df_map['group'] + 1  # 1-indexed
df_map['group_name'] = df_map['group'].apply(lambda g: f'Grupp {g}')
# Add jitter (±0.01 degrees ≈ ±1km) so overlapping kår members spread out
df_map['lat'] = df_map['lat'] + np.random.uniform(-0.01, 0.01, len(df_map))
df_map['lng'] = df_map['lng'] + np.random.uniform(-0.01, 0.01, len(df_map))

# 21 distinct colors for groups
GROUP_COLORS = [
    [31, 120, 180], [255, 127, 0], [51, 160, 44], [227, 26, 28],
    [166, 206, 227], [253, 191, 111], [178, 223, 138], [251, 154, 153],
    [202, 178, 214], [255, 255, 153], [106, 61, 154], [177, 89, 40],
    [141, 211, 199], [255, 255, 179], [190, 186, 218], [251, 128, 114],
    [128, 177, 211], [253, 180, 98], [179, 222, 105], [252, 205, 229],
    [217, 217, 217]
]

config = {
    'version': 'v1',
    'config': {
        'mapState': {
            'latitude': 60.0,
            'longitude': 15.5,
            'zoom': 5
        },
        'visState': {
            'layers': [
                {
                    'id': 'groups',
                    'type': 'point',
                    'config': {
                        'dataId': 'grupper',
                        'label': 'Deltagare',
                        'isVisible': True,
                        'columns': {'lat': 'lat', 'lng': 'lng'},
                        'color': [31, 120, 180],
                        'colorField': {'name': 'group', 'type': 'integer'},
                        'colorScale': 'ordinal',
                        'visConfig': {
                            'radius': 8,
                            'fixedRadius': False,
                            'opacity': 0.85,
                            'outline': True,
                            'thickness': 1.5,
                            'strokeColor': [255, 255, 255],
                            'colorRange': {
                                'name': 'Group Colors',
                                'type': 'qualitative',
                                'category': 'Custom',
                                'colors': ['#' + ''.join(f'{c:02x}' for c in rgb) for rgb in GROUP_COLORS]
                            },
                            'radiusRange': [4, 12],
                            'filled': True
                        }
                    }
                }
            ]
        }
    }
}

karta = KeplerGl(height=700, data={'grupper': df_map}, config=config)
karta

/usr/local/lib/python3.13/dist-packages/keplergl/keplergl.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'mapState': {'latitude': 60.0, 'longitude': 15.5, 'zoom': 5}, 'vi…

In [13]:
# Cell 11: Export map to HTML

html_path = f'{OUTPUT_DIR}/wsj_grupper_karta.html'
karta.save_to_html(file_name=html_path)

# Create fullscreen version
with open(html_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

fullscreen_css = '''
<style>
    html, body { margin: 0; padding: 0; overflow: hidden; height: 100%; width: 100%; }
    .kepler-gl, #app, #root, .map-container {
        position: absolute !important;
        top: 0; left: 0; right: 0; bottom: 0;
        width: 100vw !important;
        height: 100vh !important;
    }
</style>
'''
html_content = html_content.replace('<body>', '<body>' + fullscreen_css, 1)

with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Saved group map: {html_path}")
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {html_path}")

Map saved to /config/notebooks/wsj27/wsj_grupper_karta.html!
Saved group map: /config/notebooks/wsj27/wsj_grupper_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_grupper.json
  Map:  /config/notebooks/wsj27/wsj_grupper_karta.html
